# Notebook 03: Gold SCD2 dimensions
Stages the latest append-only Silver record per natural key, calculates SHA-256 across all configured business columns, expires changed current rows using a natural-key Delta MERGE, and appends new and changed versions. Unchanged hashes generate no write.

In [ ]:
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable
from functools import reduce
import datetime
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
SLV = f"{BRZ_SLV_LH}.slv"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
HIGH_DATE = "9999-12-31"
BATCH_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

In [ ]:
# Parameters
effective_dt = ""  # yyyy-MM-dd; blank uses the latest source EXTRACT_DATE per dimension

DIM_CONFIG = {
 "company":{"source":"t001","keys":["MANDT","BUKRS"],"columns":["MANDT","BUKRS","BUTXT","ORT01","LAND1","WAERS","KTOPL"]},
 "cost_center":{"source":"csks","keys":["MANDT","KOKRS","KOSTL"],"columns":["MANDT","KOKRS","KOSTL","BUKRS","PRCTR","VERAK","KOSAR","KHINR","DATAB","DATBI"]},
 "profit_center":{"source":"cepc","keys":["MANDT","KOKRS","PRCTR"],"columns":["MANDT","KOKRS","PRCTR","BUKRS","VERAK","KHINR","SEGMENT","DATAB","DATBI"]},
 "vendor":{"source":"lfa1","keys":["MANDT","LIFNR"],"columns":["MANDT","LIFNR","NAME1","NAME2","LAND1","ORT01","REGIO","KTOKK","STCD1"]},
 "customer":{"source":"kna1","keys":["MANDT","KUNNR"],"columns":["MANDT","KUNNR","NAME1","NAME2","LAND1","ORT01","REGIO","KTOKD","STCD1"]},
 "gl_account":{"source":"gl_account_conformed","keys":["MANDT","KTOPL","SAKNR"],"columns":["MANDT","KTOPL","SAKNR","TXT20","TXT50","XBILK","GVTYP","KTOKS"]},
 "division":{"source":"division_conformed","keys":["DIVISION_CODE"],"columns":["DIVISION_CODE","DIVISION_NAME","DATAB","DATBI"]}
}

def sha256_all(columns):
    return F.sha2(F.concat_ws("||",*[F.coalesce(F.trim(F.col(c).cast("string")),F.lit("__NULL__")) for c in columns]),256)

def latest_per_key(df,keys):
    order=[F.col("EXTRACT_DATE").desc_nulls_last(),F.col("LOAD_DATE").desc_nulls_last(),F.col("SOURCE_FILE_NAME").desc_nulls_last(),F.col("BATCH_RUN_ID").desc_nulls_last()]
    return df.withColumn("_RN",F.row_number().over(Window.partitionBy(*keys).orderBy(*order))).filter(F.col("_RN")==1).drop("_RN")

def merge_condition(keys): return " AND ".join([f"tgt.`{k}` <=> src.`{k}`" for k in keys])

def create_empty_target(target,staging,keys,effective_date):
    initial=(staging.limit(0).withColumn("EFFECTIVE_FROM",F.lit(effective_date).cast("date")).withColumn("EFFECTIVE_TO",F.lit(HIGH_DATE).cast("date")).withColumn("IS_CURRENT",F.lit(True)).withColumn("DIMENSION_SK",F.md5(F.concat_ws("||",*[F.coalesce(F.col(k).cast("string"),F.lit("__NULL__")) for k in keys],F.lit(effective_date)))).withColumn("GOLD_BATCH_ID",F.lit(BATCH_ID)).withColumn("GOLD_LOAD_TIMESTAMP",F.current_timestamp()))
    initial.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(target)

def scd2_load(dimension_name,cfg):
    source=spark.table(f"{SLV}.{cfg['source']}")
    keys=cfg['keys']; missing=[k for k in keys if k not in source.columns]
    if missing: raise ValueError(f"{dimension_name}: missing business keys {missing}")
    business_columns=[c for c in cfg['columns'] if c in source.columns]
    latest=latest_per_key(source,keys)
    staging=(latest.select(*business_columns,"BATCH_RUN_ID","EXTRACT_DATE","LOAD_DATE","SOURCE_SYSTEM","SOURCE_FILE_NAME").withColumn("RECORD_HASH",sha256_all(business_columns)))
    if staging.rdd.isEmpty(): return (dimension_name,0,0,0,0,"NO_ROWS")
    target=f"{DIM}.{dimension_name}"
    target_exists=spark.catalog.tableExists(target)
    if target_exists:
        max_extract=staging.agg(F.max("EXTRACT_DATE")).first()[0]
        effective_date=effective_dt.strip() or (max_extract.isoformat() if max_extract else datetime.date.today().isoformat())
    else:
        source_start=staging.agg(F.min("DATAB")).first()[0] if "DATAB" in staging.columns else None
        effective_date=effective_dt.strip() or (source_start.isoformat() if source_start else "1900-01-01")
    yesterday=(datetime.date.fromisoformat(effective_date)-datetime.timedelta(days=1)).isoformat()
    if not target_exists: create_empty_target(target,staging,keys,effective_date)
    current_before=spark.table(target).filter(F.col("IS_CURRENT")==True).select(*keys,F.col("RECORD_HASH").alias("TARGET_HASH"))
    classified=(staging.alias("src").join(current_before.alias("tgt"),on=keys,how="left").withColumn("CHANGE_TYPE",F.when(F.col("TARGET_HASH").isNull(),"INSERT").when(F.col("RECORD_HASH")!=F.col("TARGET_HASH"),"UPDATE").otherwise("NO_CHANGE")))
    inserts=classified.filter(F.col("CHANGE_TYPE")=="INSERT").count(); changes=classified.filter(F.col("CHANGE_TYPE")=="UPDATE").count(); unchanged=classified.filter(F.col("CHANGE_TYPE")=="NO_CHANGE").count()
    (DeltaTable.forName(spark,target).alias("tgt").merge(staging.alias("src"),f"{merge_condition(keys)} AND tgt.IS_CURRENT = true").whenMatchedUpdate(condition="tgt.RECORD_HASH <> src.RECORD_HASH",set={"IS_CURRENT":F.lit(False),"EFFECTIVE_TO":F.lit(yesterday).cast("date"),"GOLD_BATCH_ID":F.lit(BATCH_ID),"GOLD_LOAD_TIMESTAMP":F.current_timestamp()}).execute())
    current_keys=spark.table(target).filter(F.col("IS_CURRENT")==True).select(*keys).dropDuplicates(keys)
    new_versions=staging.join(current_keys,on=keys,how="left_anti")
    if not new_versions.rdd.isEmpty():
        output=(new_versions.withColumn("EFFECTIVE_FROM",F.lit(effective_date).cast("date")).withColumn("EFFECTIVE_TO",F.lit(HIGH_DATE).cast("date")).withColumn("IS_CURRENT",F.lit(True)).withColumn("DIMENSION_SK",F.md5(F.concat_ws("||",*[F.coalesce(F.col(k).cast("string"),F.lit("__NULL__")) for k in keys],F.lit(effective_date)))).withColumn("GOLD_BATCH_ID",F.lit(BATCH_ID)).withColumn("GOLD_LOAD_TIMESTAMP",F.current_timestamp()))
        output.write.format("delta").mode("append").saveAsTable(target)
    return (dimension_name,staging.count(),inserts,changes,unchanged,"SUCCESS")

# Conformed source for G/L account, using latest SKA1 and SKAT records.
ska1=latest_per_key(spark.table(f"{SLV}.ska1"),["MANDT","KTOPL","SAKNR"]).alias("a")
skat=latest_per_key(spark.table(f"{SLV}.skat"),["MANDT","SPRAS","KTOPL","SAKNR"]).filter(F.col("SPRAS")=="E").alias("t")
gl_keys=["MANDT","KTOPL","SAKNR"]
gl=(ska1.join(skat,on=gl_keys,how="left").select(*[F.col(f"a.{c}").alias(c) for c in ska1.columns],*[F.col(f"t.{c}").alias(c) for c in ["TXT20","TXT50"] if c in skat.columns]))
gl.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{SLV}.gl_account_conformed")

# Eskom division from latest CEPC.KHINR.
cepc_latest=latest_per_key(spark.table(f"{SLV}.cepc"),["MANDT","KOKRS","PRCTR"])
division=(cepc_latest.select(F.col("KHINR").alias("DIVISION_CODE"),F.col("SEGMENT").alias("DIVISION_NAME"),"DATAB","DATBI","BATCH_RUN_ID","EXTRACT_DATE","LOAD_DATE","SOURCE_SYSTEM","SOURCE_FILE_NAME").filter(F.col("DIVISION_CODE").isNotNull()).dropDuplicates(["DIVISION_CODE"]))
division.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{SLV}.division_conformed")

results=[scd2_load(name,cfg) for name,cfg in DIM_CONFIG.items()]
run_log=(spark.createDataFrame(results,"dimension_name string, staged_rows long, inserted_keys long, changed_keys long, unchanged_keys long, status string").withColumn("gold_batch_id",F.lit(BATCH_ID)).withColumn("gold_load_timestamp",F.current_timestamp()))
run_log.write.mode("append").format("delta").saveAsTable(f"{AUD}.gold_scd2_run_log")
display(run_log)

# Date dimension. Calendar fiscal mapping is a demo assumption and must be replaced with Eskom's confirmed fiscal calendar.
dates=spark.sql("SELECT explode(sequence(to_date('2018-01-01'),to_date('2035-12-31'),interval 1 day)) AS CALENDAR_DATE")
dates=(dates.withColumn("DATE_SK",F.date_format("CALENDAR_DATE","yyyyMMdd").cast("int")).withColumn("CALENDAR_YEAR",F.year("CALENDAR_DATE")).withColumn("QUARTER_NUMBER",F.quarter("CALENDAR_DATE")).withColumn("MONTH_NUMBER",F.month("CALENDAR_DATE")).withColumn("MONTH_NAME",F.date_format("CALENDAR_DATE","MMMM")).withColumn("FISCAL_YEAR",F.year("CALENDAR_DATE")).withColumn("FISCAL_PERIOD",F.month("CALENDAR_DATE")))
dates.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{DIM}.date")

# Fiscal period dimension supports SAP normal periods 1-12 and special periods 13-16.
fiscal_years = spark.range(2018, 2036).select(
    F.col("id").cast("int").alias("FISCAL_YEAR")
)
fiscal_period_numbers = spark.range(1, 17).select(
    F.col("id").cast("int").alias("FISCAL_PERIOD")
)
fiscal_periods = (
    fiscal_years.crossJoin(fiscal_period_numbers)
    .withColumn(
        "FISCAL_PERIOD_KEY",
        (F.col("FISCAL_YEAR") * F.lit(100) + F.col("FISCAL_PERIOD")).cast("int"),
    )
    .withColumn(
        "FISCAL_PERIOD_NAME",
        F.when(
            F.col("FISCAL_PERIOD") <= 12,
            F.concat(F.lit("Period "), F.lpad(F.col("FISCAL_PERIOD"), 2, "0")),
        ).otherwise(
            F.concat(
                F.lit("Special Period "),
                F.lpad(F.col("FISCAL_PERIOD"), 2, "0"),
            )
        ),
    )
    .withColumn(
        "FISCAL_QUARTER",
        F.when(
            F.col("FISCAL_PERIOD") <= 12,
            F.concat(
                F.lit("Q"),
                F.ceil(F.col("FISCAL_PERIOD") / F.lit(3)).cast("int").cast("string"),
            ),
        ).otherwise(F.lit("Special")),
    )
    .withColumn("IS_SPECIAL_PERIOD", F.col("FISCAL_PERIOD") > 12)
    .withColumn(
        "FISCAL_YEAR_PERIOD_LABEL",
        F.concat(
            F.col("FISCAL_YEAR").cast("string"),
            F.lit("-P"),
            F.lpad(F.col("FISCAL_PERIOD"), 2, "0"),
        ),
    )
    .withColumn("FISCAL_YEAR_PERIOD_SORT", F.col("FISCAL_PERIOD_KEY"))
    .select(
        "FISCAL_PERIOD_KEY",
        "FISCAL_YEAR",
        "FISCAL_PERIOD",
        "FISCAL_PERIOD_NAME",
        "FISCAL_QUARTER",
        "IS_SPECIAL_PERIOD",
        "FISCAL_YEAR_PERIOD_LABEL",
        "FISCAL_YEAR_PERIOD_SORT",
    )
)
fiscal_periods.write.mode("overwrite").option(
    "overwriteSchema", "true"
).format("delta").saveAsTable(f"{DIM}.fiscal_period")
